# Phase 3 Validation (Bottleneck / Duration)

Manual validation notebook for `pm_spark.bottleneck.duration` using `create_sample_event_log()`.

**Covered functions**
- `waiting_time_between_activities()`
- `time_between_activity_occurrences()`
- `activity_transition_matrix()`
- `transition_time_matrix()`
- `longest_waiting_step_per_case()`
- `time_to_first_occurrence_from_case_start()`
- `parallel_activity_flag()`
- `activity_loop_detection()`

**Notes (current `duration.py`)**
- Timestamp handling and shared column helpers live in `pm_spark._common` (includes `TimestampNTZType`).
- Optional `tiebreak_col` on waiting-time helpers and transition/loop functions for stable ordering when timestamp + activity tie.
- `activity_transition_matrix` / `transition_time_matrix`: `sort_result=True` (default) applies the result `orderBy`; set `False` to skip if you sort downstream.
- `parallel_activity_flag`: **intra-case** same instant — more than one event at the same `timestamp` within a case (sample: C005 has B and P at 09:05). Cross-case identical timestamps are not flagged. Only cases with at least one non-null `timestamp` appear in the output.
- `transition_time_matrix` median uses `percentile_approx(..., 0.50, accuracy)` with a plain Python `accuracy` value.

In [1]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Ensure project root is importable when notebook runs from dev/
project_root = Path.cwd().resolve()
if project_root.name == "dev":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("pm_spark_phase3_validation").getOrCreate()

from dev.sample_data import create_sample_event_log
from pm_spark.bottleneck.duration import (
    activity_loop_detection,
    activity_transition_matrix,
    longest_waiting_step_per_case,
    parallel_activity_flag,
    time_between_activity_occurrences,
    time_to_first_occurrence_from_case_start,
    transition_time_matrix,
    waiting_time_between_activities,
)

def show_df(df, n=200):
    df.show(n, truncate=False)

26/03/22 11:30:33 WARN Utils: Your hostname, MacBook-Air-von-Alex.local resolves to a loopback address: 127.0.0.1; using 192.168.178.137 instead (on interface en0)
26/03/22 11:30:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 11:30:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/22 11:30:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
event_log = create_sample_event_log(spark)
show_df(event_log.orderBy("case_key", "timestamp", "activity"))

+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |A            |2

In [3]:
# 3.1 waiting_time_between_activities() -> returns (event_level_df, summary_df)
df_wait_event, df_wait_summary = waiting_time_between_activities(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    unit="minutes",
    summary_aggs=["avg", "sum", "max"],
)
show_df(df_wait_event.orderBy("case_key", "timestamp", "activity"))
show_df(df_wait_summary.orderBy("case_key"))

# Expected checks:
# - C001 has waits [5, 5, 10] minutes -> avg=6.67, sum=20, max=10
# - C004 includes a 3-minute B->B wait

+--------+-------------+-------------------+-------------+-------------------+------------+
|case_key|activity     |timestamp          |prev_activity|prev_timestamp     |waiting_time|
+--------+-------------+-------------------+-------------+-------------------+------------+
|C001    |B            |2026-03-18 09:05:00|A            |2026-03-18 09:00:00|5.0         |
|C001    |C            |2026-03-18 09:10:00|B            |2026-03-18 09:05:00|5.0         |
|C001    |D            |2026-03-18 09:20:00|C            |2026-03-18 09:10:00|10.0        |
|C002    |B            |2026-03-18 09:04:00|A            |2026-03-18 09:00:00|4.0         |
|C002    |Manual_Review|2026-03-18 09:08:00|B            |2026-03-18 09:04:00|4.0         |
|C002    |C            |2026-03-18 09:15:00|Manual_Review|2026-03-18 09:08:00|7.0         |
|C002    |D            |2026-03-18 09:25:00|C            |2026-03-18 09:15:00|10.0        |
|C003    |C            |2026-03-18 10:10:00|A            |2026-03-18 10:00:00|10

In [4]:
# 3.2 time_between_activity_occurrences()
df_a_to_d_first_first = time_between_activity_occurrences(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    activity_a="A",
    activity_b="D",
    anchor_a="first",
    anchor_b="first",
    unit="minutes",
)
show_df(df_a_to_d_first_first.orderBy("case_key"))

df_a_to_a_last_last = time_between_activity_occurrences(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    activity_a="A",
    activity_b="A",
    anchor_a="last",
    anchor_b="last",
    unit="minutes",
)
show_df(df_a_to_a_last_last.orderBy("case_key"))

# Expected checks:
# - C001 A(first)->D(first) = 20 minutes
# - C006 A(last)->A(last) = 0 minutes

+--------+--------------------+--------------------+--------+
|case_key|activity_a_timestamp|activity_b_timestamp|duration|
+--------+--------------------+--------------------+--------+
|C001    |2026-03-18 09:00:00 |2026-03-18 09:20:00 |20.0    |
|C002    |2026-03-18 09:00:00 |2026-03-18 09:25:00 |25.0    |
|C003    |2026-03-18 10:00:00 |2026-03-18 10:30:00 |30.0    |
|C004    |2026-03-18 11:00:00 |2026-03-18 11:30:00 |30.0    |
|C005    |2026-03-19 09:00:00 |2026-03-19 09:20:00 |20.0    |
|C006    |2026-03-19 09:00:00 |2026-03-19 09:30:00 |30.0    |
|C007    |2026-03-19 12:00:00 |2026-03-19 12:25:00 |25.0    |
+--------+--------------------+--------------------+--------+

+--------+--------------------+--------------------+--------+
|case_key|activity_a_timestamp|activity_b_timestamp|duration|
+--------+--------------------+--------------------+--------+
|C001    |2026-03-18 09:00:00 |2026-03-18 09:00:00 |0.0     |
|C002    |2026-03-18 09:00:00 |2026-03-18 09:00:00 |0.0     |
|C003  

In [5]:
# 3.3 activity_transition_matrix()
df_transition_counts = activity_transition_matrix(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    transition_count_col="transition_count",
)
show_df(df_transition_counts)

# Expected checks:
# - Transition A->B appears in multiple cases
# - Self-transition B->B appears for C004

+-------------+-------------+----------------+
|from_activity|to_activity  |transition_count|
+-------------+-------------+----------------+
|C            |D            |7               |
|A            |B            |6               |
|B            |C            |3               |
|A            |C            |2               |
|B            |A            |1               |
|B            |B            |1               |
|B            |Manual_Review|1               |
|B            |P            |1               |
|Manual_Review|C            |1               |
|P            |C            |1               |
+-------------+-------------+----------------+



In [6]:
# 3.4 transition_time_matrix()
df_transition_time = transition_time_matrix(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    unit="minutes",
    accuracy=10000,
    avg_col="avg_transition_time",
    median_col="median_transition_time",
    count_col="transition_count",
)
show_df(df_transition_time)

# Expected checks:
# - Columns include transition_count, avg_transition_time, median_transition_time
# - Durations are non-negative and rounded to 2 decimals

+-------------+-------------+----------------+-------------------+----------------------+
|from_activity|to_activity  |transition_count|avg_transition_time|median_transition_time|
+-------------+-------------+----------------+-------------------+----------------------+
|C            |D            |7               |11.29              |10.0                  |
|A            |B            |6               |5.0                |5.0                   |
|B            |C            |3               |8.33               |10.0                  |
|A            |C            |2               |8.0                |6.0                   |
|B            |A            |1               |6.0                |6.0                   |
|B            |B            |1               |3.0                |3.0                   |
|B            |Manual_Review|1               |4.0                |4.0                   |
|B            |P            |1               |0.0                |0.0                   |
|Manual_Re

In [7]:
# 3.5 longest_waiting_step_per_case()
df_longest_wait = longest_waiting_step_per_case(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    unit="minutes",
    output_col="longest_waiting_time",
)
show_df(df_longest_wait.orderBy("case_key"))

# Expected checks:
# - C001 longest_waiting_time = 10.0
# - C003 longest_waiting_time = 20.0

+--------+--------------------+
|case_key|longest_waiting_time|
+--------+--------------------+
|C001    |10.0                |
|C002    |10.0                |
|C003    |20.0                |
|C004    |12.0                |
|C005    |10.0                |
|C006    |12.0                |
|C007    |10.0                |
+--------+--------------------+



In [8]:
# 3.6 time_to_first_occurrence_from_case_start()
df_time_to_c = time_to_first_occurrence_from_case_start(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    target_activity="C",
    unit="minutes",
    output_col="time_to_first_c",
)
show_df(df_time_to_c.orderBy("case_key"))

# Expected checks:
# - C001 time_to_first_c = 10.0
# - All shown cases contain activity C

+--------+---------------+
|case_key|time_to_first_c|
+--------+---------------+
|C001    |10.0           |
|C002    |15.0           |
|C003    |10.0           |
|C004    |18.0           |
|C005    |15.0           |
|C006    |18.0           |
|C007    |15.0           |
+--------+---------------+



In [9]:
# 3.7 parallel_activity_flag() — intra-case: >1 event at the same timestamp
df_parallel = parallel_activity_flag(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="is_parallel",
)
show_df(df_parallel.orderBy("case_key"))

# Expected checks:
# - C005 -> True (activities B and P share 2026-03-19 09:05:00 in the sample)
# - C001/C002 -> False (shared 09:00 across cases is not intra-case parallelism)
# - One row per case that has at least one non-null timestamp (no extra scan/join)

+--------+-----------+
|case_key|is_parallel|
+--------+-----------+
|C001    |false      |
|C002    |false      |
|C003    |false      |
|C004    |false      |
|C005    |true       |
|C006    |false      |
|C007    |false      |
+--------+-----------+



In [10]:
# 3.8 activity_loop_detection()
df_loops = activity_loop_detection(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    loop_count_col="loop_count",
    loop_pairs_col="loop_pairs",
)
show_df(df_loops.orderBy("case_key"))

# Expected checks:
# - C004 should include B→B (self-loop)
# - C006 should include A→B→A (direct loop)

+--------+----------+----------+
|case_key|loop_count|loop_pairs|
+--------+----------+----------+
|C004    |1         |B→B       |
|C006    |1         |A→B→A     |
+--------+----------+----------+



## Validation Checklist (Phase 3)

- [ ] 3.1 `waiting_time_between_activities()` validated
- [ ] 3.2 `time_between_activity_occurrences()` validated
- [ ] 3.3 `activity_transition_matrix()` validated
- [ ] 3.4 `transition_time_matrix()` validated
- [ ] 3.5 `longest_waiting_step_per_case()` validated
- [ ] 3.6 `time_to_first_occurrence_from_case_start()` validated
- [ ] 3.7 `parallel_activity_flag()` validated
- [ ] 3.8 `activity_loop_detection()` validated